# BSD35k Class-Selective Augmentation 실험

목표: **BSD35k 증강으로 BSD10k baseline(H-Acc 79.24%)을 *넘는* 것.**

근거 분석: `deft논문기반_실험/bsd35k_filtering_replan_ko.md`

핵심 아이디어 — 전역 v4 threshold(ge4)는 baseline에 아래에서 수렴할 뿐 넘지 못한다.
대신 **per-class 증강 delta**(35k가 그 클래스를 돕는지/해치는지 실측값)로
- 해치는 클래스(fx-n, fx-m 등 Δ<-1)는 35k를 **빼고**,
- 돕는 클래스(m-m, fx-el 등 Δ≥+1)는 35k를 **넣는다**.

| 전략 | 정의 |
|---|---|
| `baseline_b0` | BSD10k train80만 (이 노트북에서 직접 재학습) |
| `C1` | Δ<-1 클래스 DROP, 나머지 ge4 |
| `C2` | Δ<-1 클래스 DROP, Δ≥+1 클래스 ge2(데이터 최대), 중립 클래스 ge4 |
| (참고) `v4_ge4`, `v4_all` | 기존 `v4_35k_baseline_model` 결과를 비교표에 병기 |

**주의**: 기존 실험과 동일한 holdout·seed를 쓰기 위해 `SEED=1821`을 유지한다. 변경 금지.


## 0. Setup

In [ ]:
from pathlib import Path
import json
import glob
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import StratifiedKFold

HERE = Path.cwd()
if not (HERE / 'baseline_confidnce_train').exists():
    ROOT = HERE.parent
else:
    ROOT = HERE

sys.path.insert(0, str(ROOT / 'baseline_confidnce_train'))
sys.path.insert(0, str(ROOT / 'dcase2026_task1_baseline'))

import confidence_baseline_common as cbc

# 기존 v4_35k_baseline_model 실험과 동일하게 유지 (holdout 재현 위해 필수)
SEED = 1821
BASELINE_MODES = ('both',)
N_FOLDS = 5
NUM_EPOCHS = 100
BATCH_SIZE = 64

OUTPUT_ROOT = ROOT / 'baseline_confidnce_train' / 'outputs' / 'v4_35k_class_selective'
PLOTS_DIR = OUTPUT_ROOT / 'plots'
DATASET_DIR = OUTPUT_ROOT / 'datasets'
for path in [OUTPUT_ROOT, PLOTS_DIR, DATASET_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# 기존 전역-threshold 실험 결과 폴더 (비교표 + per-class delta 근거)
PREV_AUG_ROOT = ROOT / 'baseline_confidnce_train' / 'outputs' / 'v4_35k_baseline_model'
BASELINE_PROXY_ROOT = ROOT / 'baseline_confidnce_train' / 'outputs' / '517_mlp_classification'

random.seed(SEED)
np.random.seed(SEED)
cbc.seed_everything(SEED)

print('ROOT:', ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('device:', cbc.device_summary())

## 1. Fixed BSD10k 80/20 split (기존과 동일 seed → 동일 holdout)

In [ ]:
full_df, class_dict, top_class_dict = cbc.load_baseline_assets(ROOT)
train_pool, final_test, split_df = cbc.make_fixed_holdout(full_df, OUTPUT_ROOT, seed=SEED, test_size=0.2)

print('BSD10k full:', len(full_df))
print('BSD10k train_pool 80%:', len(train_pool))
print('BSD10k final_test 20%:', len(final_test))
print('classes in train_pool:', train_pool['class'].nunique())
display(split_df.head())

## 2. Load BSD35k v4 predictions -> DCASE-compatible rows

`v4_35k_baseline모델.ipynb`의 `build_bsd35k_dcase_dataframe`와 동일.

In [ ]:
BSD35K_V4_PATH = ROOT / 'outputs' / 'confidence_filter_v4' / 'predictions' / 'BSD35k-CS_filter_predictions_v4.csv'
assert BSD35K_V4_PATH.exists(), f'Missing file: {BSD35K_V4_PATH}'

def build_bsd35k_dcase_dataframe(v4_path: Path) -> pd.DataFrame:
    score_df = pd.read_csv(v4_path)
    score_df['sound_id'] = score_df['sound_id'].astype(str)
    top_col = 'class_top' if 'class_top' in score_df.columns else 'top_class'
    needed = ['sound_id', 'class', top_col, 'v4_filter_score', 'binary_mlp_prob', 'fiveclass_score', 'fiveclass_p45']
    missing = [c for c in needed if c not in score_df.columns]
    if missing:
        raise ValueError(f'Missing columns in BSD35k v4 file: {missing}')
    out = score_df[needed].copy().rename(columns={top_col: 'top_class'})
    out['index'] = out['sound_id'].astype(str)
    out['class'] = out['class'].astype(str)
    out['top_class'] = out['top_class'].astype(str)
    out['class_idx'] = out['class'].map(class_dict)
    out['top_class_idx'] = out['top_class'].map(top_class_dict)
    bsd35k_audio_dir = ROOT / 'data' / 'features' / 'BSD35k_clap_audio_embeddings'
    bsd35k_text_dir = ROOT / 'data' / 'features' / 'BSD35k-CS_clap_text_embeddings'
    out['audio_emb_filepath'] = out['index'].map(lambda sid: str(bsd35k_audio_dir / f'{sid}.npy'))
    out['text_emb_filepath'] = out['index'].map(lambda sid: str(bsd35k_text_dir / f'{sid}.npy'))
    out['predicted_confidence_score'] = 1.0 + 4.0 * out['v4_filter_score'].astype(float)
    before = len(out)
    bsd10k_ids = set(full_df['index'].astype(str))
    out = out[~out['index'].isin(bsd10k_ids)].copy()
    out = out[out['class_idx'].notna() & out['top_class_idx'].notna()].copy()
    out = out[out['audio_emb_filepath'].map(lambda p: Path(p).exists())].copy()
    out = out[out['text_emb_filepath'].map(lambda p: Path(p).exists())].copy()
    out['class_idx'] = out['class_idx'].astype(int)
    out['top_class_idx'] = out['top_class_idx'].astype(int)
    out = out.reset_index(drop=True)
    print(f'BSD35k rows before filtering: {before:,}  ->  usable: {len(out):,}  | classes: {out["class"].nunique()}')
    return out

bsd35k_all = build_bsd35k_dcase_dataframe(BSD35K_V4_PATH)
display(bsd35k_all.head(3))

## 3. per-class 증강 delta 계산 -> class policy

기존 결과에서 **클래스별 recall**을 읽어 delta를 만든다.
- baseline = `517_mlp_classification/pred_ge_2` (retained 100% = BSD10k-only)
- best 전역 증강 = `v4_35k_baseline_model/v4_ge4`

정책:
- `delta_ge4 < -1.0` -> **DROP**
- `-1.0 <= delta_ge4 < +1.0` -> **ge4**
- `delta_ge4 >= +1.0` -> C1: ge4 / C2: ge2

In [ ]:
def mean_diag(pattern):
    mats = []
    for f in sorted(glob.glob(pattern)):
        df = pd.read_csv(f, index_col=0)
        mats.append(pd.Series(np.diag(df.values), index=df.index))
    if not mats:
        raise FileNotFoundError(f'No confusion matrices: {pattern}')
    return pd.concat(mats, axis=1).mean(axis=1) * 100

base_recall = mean_diag(str(BASELINE_PROXY_ROOT / 'pred_ge_2' / 'both' / 'fold_*' / 'evaluation' / 'confusion_matrix_normalized_true.csv'))
ge4_recall  = mean_diag(str(PREV_AUG_ROOT / 'v4_ge4' / 'both' / 'fold_*' / 'evaluation' / 'confusion_matrix_normalized_true.csv'))

delta = pd.DataFrame({'baseline_recall': base_recall.round(2), 'v4_ge4_recall': ge4_recall.round(2)})
delta['delta_ge4'] = (delta['v4_ge4_recall'] - delta['baseline_recall']).round(2)
delta.index.name = 'class'

def tier_of(d):
    if d < -1.0: return 'DROP'
    if d < 1.0:  return 'ge4'
    return 'helped'
delta['policy_tier'] = delta['delta_ge4'].apply(tier_of)

# strategy thresholds on predicted_confidence_score (1 + 4*v4_filter_score)
def thr(tier, strat):
    if tier == 'DROP':   return None
    if tier == 'ge4':    return 4.0
    return 4.0 if strat == 'C1' else 2.0   # helped
for strat in ['C1', 'C2']:
    delta[f'{strat}_threshold'] = delta['policy_tier'].apply(lambda t: thr(t, strat))

delta['n_35k_total'] = delta.index.map(bsd35k_all.groupby('class').size()).fillna(0).astype(int)
display(delta.sort_values('delta_ge4'))
print('DROP classes:', delta[delta.policy_tier == 'DROP'].index.tolist())

## 4. build C1 / C2 addition sets (+ subset & retained CSVs)

In [ ]:
def build_subset(strat):
    parts = []
    for cls, row in delta.iterrows():
        t = row[f'{strat}_threshold']
        if t is None or pd.isna(t):
            continue
        parts.append(bsd35k_all[(bsd35k_all['class'] == cls) & (bsd35k_all['predicted_confidence_score'] >= t)])
    return pd.concat(parts, ignore_index=True).reset_index(drop=True) if parts else bsd35k_all.iloc[0:0]

C1 = build_subset('C1')
C2 = build_subset('C2')

# downstream-ready subset CSV (key cols)
keepcols = ['sound_id', 'class', 'top_class', 'v4_filter_score', 'binary_mlp_prob', 'fiveclass_score', 'predicted_confidence_score']
C1[keepcols].assign(strategy='C1').to_csv(OUTPUT_ROOT / 'bsd35k_subset_C1_classselective.csv', index=False)
C2[keepcols].assign(strategy='C2').to_csv(OUTPUT_ROOT / 'bsd35k_subset_C2_classselective.csv', index=False)

# class retained analysis
ret = delta.copy()
for name, sub in [('C1', C1), ('C2', C2)]:
    cnt = sub.groupby('class').size()
    ret[f'{name}_retained'] = ret.index.map(cnt).fillna(0).astype(int)
    ret[f'{name}_ratio'] = (ret[f'{name}_retained'] / ret['n_35k_total'].replace(0, np.nan)).round(3)
ret.to_csv(OUTPUT_ROOT / 'bsd35k_class_retained_analysis.csv')
delta.to_csv(OUTPUT_ROOT / 'per_class_aug_delta.csv')

print(f'C1: {len(C1):,} added  ({len(C1)/len(bsd35k_all)*100:.1f}%)  classes={C1["class"].nunique()}')
print(f'C2: {len(C2):,} added  ({len(C2)/len(bsd35k_all)*100:.1f}%)  classes={C2["class"].nunique()}')
display(ret[['delta_ge4', 'policy_tier', 'n_35k_total', 'C1_retained', 'C2_retained']].sort_values('delta_ge4'))

## 5. Train (baseline_b0, C1, C2)

같은 holdout(final_test 2192)에서 평가. 기존에 학습된 run이 있으면 자동 skip.
`RUN_DATASET_LABELS`를 편집해 일부만 돌릴 수 있다.

In [ ]:
addition_sets = {
    'baseline_b0': bsd35k_all.iloc[0:0],   # BSD10k train80만
    'C1': C1,
    'C2': C2,
}

RUN_DATASET_LABELS = ['baseline_b0', 'C1', 'C2']

def make_combined_dataset(addition_df):
    if len(addition_df) == 0:
        return train_pool.reset_index(drop=True).copy()
    return pd.concat([train_pool, addition_df[train_pool.columns]], ignore_index=True).reset_index(drop=True)

def run_one_dataset(dataset_label, combined_df):
    rows = []
    dataset_dir = OUTPUT_ROOT / dataset_label
    dataset_dir.mkdir(parents=True, exist_ok=True)
    combined_df.to_csv(dataset_dir / 'combined_train_pool.csv', index=False)
    splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    splits = list(splitter.split(np.zeros(len(combined_df)), combined_df['class_idx']))
    for mode in BASELINE_MODES:
        for fold, (train_idx, val_idx) in enumerate(splits):
            run_dir = dataset_dir / mode / f'fold_{fold}'
            tr_df = combined_df.iloc[train_idx].reset_index(drop=True)
            va_df = combined_df.iloc[val_idx].reset_index(drop=True)
            base_row = {
                'dataset_label': dataset_label, 'mode': mode, 'split': f'fold_{fold}',
                'train_samples': int(len(tr_df)), 'val_samples': int(len(va_df)),
                'final_test_samples': int(len(final_test)),
                'bsd10k_train80_samples': int(len(train_pool)),
                'added_bsd35k_samples': int(max(len(combined_df) - len(train_pool), 0)),
                'output_dir': str(run_dir),
            }
            metrics_path = run_dir / 'evaluation' / 'results.txt'
            if metrics_path.exists():
                rows.append({**base_row, 'status': 'completed_existing', **cbc.read_metrics_file(metrics_path)})
                print('[skip] existing run:', run_dir)
                continue
            best_val_acc, metrics = cbc.train_and_evaluate_one(
                tr_df, va_df, final_test, class_dict, top_class_dict, run_dir,
                mode=mode, seed=SEED, batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS,
                lr=0.001, patience=5, early_stopping_factor=3,
            )
            rows.append({**base_row, 'status': 'ok', 'best_val_accuracy': float(best_val_acc), **metrics})
            pd.DataFrame(rows).to_csv(dataset_dir / 'summary_results_partial.csv', index=False)
    return rows

all_rows = []
for dataset_label in RUN_DATASET_LABELS:
    print('\n' + '=' * 90)
    print('Running dataset:', dataset_label)
    rows = run_one_dataset(dataset_label, make_combined_dataset(addition_sets[dataset_label]))
    all_rows.extend(rows)
    pd.DataFrame(all_rows).to_csv(OUTPUT_ROOT / 'summary_results.csv', index=False)

summary = pd.DataFrame(all_rows)
summary.to_csv(OUTPUT_ROOT / 'summary_results.csv', index=False)
display(summary)

## 6. 전략 비교표 (baseline_b0 / C1 / C2 + 기존 v4_ge4·v4_all)

성공 기준: **H-Acc > baseline_b0**

In [ ]:
METRICS = ['accuracy', 'hierarchical_accuracy', 'hierarchical_f1', 'macro_accuracy', 'macro_top_accuracy', 'top_accuracy']

def mean_std(df):
    out = {}
    for m in METRICS:
        if m in df.columns:
            out[f'{m}_mean'] = round(float(df[m].mean()), 3)
            out[f'{m}_std'] = round(float(df[m].std()), 3)
    return out

rows = []
for lab, g in summary.groupby('dataset_label'):
    rows.append({'strategy': lab, 'added_bsd35k': int(g['added_bsd35k_samples'].iloc[0]), **mean_std(g)})

# 기존 전역-threshold 실험 병기 (있으면)
prev_path = PREV_AUG_ROOT / 'fold_metric_summary_mean_std.csv'
if prev_path.exists():
    prev = pd.read_csv(prev_path)
    for _, r in prev.iterrows():
        row = {'strategy': r['dataset_label'], 'added_bsd35k': '-'}
        for m in METRICS:
            if f'{m}_mean' in r:
                row[f'{m}_mean'] = round(float(r[f'{m}_mean']), 3)
                row[f'{m}_std'] = round(float(r[f'{m}_std']), 3)
        rows.append(row)

comp = pd.DataFrame(rows).sort_values('hierarchical_accuracy_mean', ascending=False).reset_index(drop=True)
comp.to_csv(OUTPUT_ROOT / 'aug_strategy_comparison.csv', index=False)
display(comp[['strategy', 'added_bsd35k', 'accuracy_mean', 'hierarchical_accuracy_mean', 'macro_accuracy_mean', 'top_accuracy_mean']])

if 'baseline_b0' in comp['strategy'].values:
    b0 = comp.loc[comp['strategy'] == 'baseline_b0', 'hierarchical_accuracy_mean'].iloc[0]
    print(f'
baseline_b0 H-Acc = {b0:.3f}')
    for lab in ['C1', 'C2']:
        if lab in comp['strategy'].values:
            v = comp.loc[comp['strategy'] == lab, 'hierarchical_accuracy_mean'].iloc[0]
            verdict = 'BEATS baseline' if v > b0 else 'below baseline'
            print(f'{lab} H-Acc = {v:.3f}  ({v - b0:+.3f})  -> {verdict}')

## 7. per-class recall 변화 (baseline_b0 vs C1 vs C2)

In [ ]:
def diag_for(label):
    pat = str(OUTPUT_ROOT / label / 'both' / 'fold_*' / 'evaluation' / 'confusion_matrix_normalized_true.csv')
    files = sorted(glob.glob(pat))
    if not files:
        return None
    mats = []
    for f in files:
        df = pd.read_csv(f, index_col=0)
        mats.append(pd.Series(np.diag(df.values), index=df.index))
    return pd.concat(mats, axis=1).mean(axis=1) * 100

pc = pd.DataFrame()
for lab in ['baseline_b0', 'C1', 'C2']:
    d = diag_for(lab)
    if d is not None:
        pc[lab] = d.round(2)
if 'baseline_b0' in pc.columns:
    for lab in ['C1', 'C2']:
        if lab in pc.columns:
            pc[f'{lab}_minus_base'] = (pc[lab] - pc['baseline_b0']).round(2)
    if 'C1_minus_base' in pc.columns:
        pc = pc.sort_values('C1_minus_base')
pc['policy_tier'] = pc.index.map(delta['policy_tier'])
pc.to_csv(OUTPUT_ROOT / 'per_class_recall_change.csv')
display(pc)